# Phase 2 — 유튜브 댓글 & 자막(STT) 수집

## 수집 전략: 하이브리드 방식

앱스토어와 달리 유튜브는 특정 서비스 리뷰가 한 곳에 모여있지 않습니다.
따라서 **두 가지 방식을 병행**합니다:

| 방식 | 설명 | 장점 |
|------|------|------|
| ① **직접 지정** | WBS 리서치 과정에서 이미 확인한 핵심 영상 ID 직접 입력 | 높은 정확도, 관련성 보장 |
| ② **키워드 검색** | 검색어로 추가 영상 발굴 (수동 필터링 필요) | 새로운 영상 발굴 가능 |

**수집 내용**
- `youtube_videos`: 영상 메타데이터 (제목, 채널, 조회수 등)
- `youtube_comments`: 영상 댓글
- `youtube_stt`: 유튜브 자동 자막

⚠️ **라오어 자막 주의**: 자동 생성이 없거나 품질이 낮을 수 있음

In [ ]:
# 필요 라이브러리 설치
# !pip install google-api-python-client youtube-transcript-api psycopg2-binary pandas python-dotenv

In [ ]:
import sys
sys.path.append('../')

import os
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled
from src.db import insert_df, upsert_df

load_dotenv(dotenv_path='../.env')
YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

## ① 직접 지정 영상 (WBS 리서치에서 확인된 핵심 영상)

In [ ]:
# WBS 데이터수집 시트에서 직접 확인한 핵심 영상
# 관련성이 보장된 영상들 — 수동으로 검토 완료
CURATED_VIDEOS = [
    {
        "video_id": "cuFkQBe26mM",
        "country":  "LAO",
        "note":     "라오스 LOCA EV 플랫폼 리얼 사용기 (충전 프로세스, QR 오류, 결제)"
    },
    {
        "video_id": "nCMVe_m-PTc",
        "country":  "LAO",
        "note":     "라오스 EV 충전 인프라 기업 Loca EV — 1년 만에 투자금 회수 성공 스토리"
    },
    {
        "video_id": "6rw0NJdxgMs",
        "country":  "LAO",
        "note":     "라오스 전기 오토바이 여행 — 충전소 없어 카페 콘센트 사용 (인프라 부재)"
    },
    {
        "video_id": "nOpmGFkntb4",
        "country":  "LAO",
        "note":     "라오스 루앙프라방 소형 전기차 렌트 브이로그 (배터리 효율, 충전 위치)"
    },
    {
        "video_id": "Jja5en9n9pw",
        "country":  "LAO",
        "note":     "라오스 EV 충전 서비스 현황 — LOCA EV 17개 급속 스테이션 운영"
    },
    {
        "video_id": "moNqhkSmNNg",
        "country":  "THA",
        "note":     "태국 5대 EV 충전 네트워크 비교 (PTT PluZ, Elex, SPARK, PEA Volta, Tesla)"
    },
]

print(f"직접 지정 영상: {len(CURATED_VIDEOS)}개")
for v in CURATED_VIDEOS:
    print(f"  - {v['video_id']} [{v['country']}]: {v['note'][:50]}...")

## ② 키워드 검색 영상 (추가 발굴)

In [ ]:
# 추가 발굴용 검색어
# ⚠️ 검색 결과는 수동 필터링 후 CURATED_VIDEOS에 추가 권장
SEARCH_QUERIES = [
    {"query": "LOCA EV charging Laos",       "country": "LAO"},
    {"query": "LokaEV ລາວ ການສາກໄຟ",         "country": "LAO"},  # 라오어
    {"query": "Green SM EV Laos review",      "country": "LAO"},
    {"query": "V-Green EV Vietnam sạc điện",  "country": "VNM"},  # 베트남어 혼합
    {"query": "라오스 전기차 충전 리뷰",        "country": "KOR"},  # 한국인 여행객
    {"query": "라오스 EV 충전소 여행",          "country": "KOR"},
]

MAX_RESULTS_PER_QUERY = 10  # 검색당 최대 영상 수 (필터링 후 실제 사용 수 줄어듦)

In [ ]:
def search_videos(query: str, country: str, max_results: int = 10) -> list:
    """키워드 검색으로 video_id 목록 반환"""
    print(f"\n🔍 검색: '{query}'")
    try:
        resp = youtube.search().list(
            q=query, part='id', type='video',
            maxResults=max_results, order='relevance'
        ).execute()
        ids = [item['id']['videoId'] for item in resp.get('items', [])]
        print(f"  → {len(ids)}개 발견: {ids}")
        return [{"video_id": vid, "country": country, "note": f"검색: {query}"} for vid in ids]
    except Exception as e:
        print(f"  ❌ 오류: {e}")
        return []

# 검색 실행
search_results = []
for item in SEARCH_QUERIES:
    results = search_videos(item['query'], item['country'], MAX_RESULTS_PER_QUERY)
    search_results.extend(results)

print(f"\n🔍 검색으로 발굴된 영상: {len(search_results)}개")
print("⚠️ 아래 video_id를 검토 후 관련 있는 것만 CURATED_VIDEOS에 추가하세요.")

## 최종 수집 대상 통합 (중복 제거)

In [ ]:
# 직접 지정 + 검색 결과 통합 (중복 제거)
all_targets = CURATED_VIDEOS + search_results

# video_id 기준 중복 제거 (직접 지정 우선)
seen = set()
unique_targets = []
for t in all_targets:
    if t['video_id'] not in seen:
        seen.add(t['video_id'])
        unique_targets.append(t)

print(f"\n📋 최종 수집 대상: {len(unique_targets)}개")
print(f"  - 직접 지정: {len(CURATED_VIDEOS)}개")
print(f"  - 검색 추가: {len(unique_targets) - len(CURATED_VIDEOS)}개")

## 영상 메타데이터 수집 → youtube_videos

In [ ]:
def fetch_video_metadata(targets: list) -> pd.DataFrame:
    """video_id 목록으로 영상 메타데이터 수집"""
    # 국가 정보 맵 생성
    country_map = {t['video_id']: t['country'] for t in targets}
    video_ids = list(country_map.keys())

    # 50개씩 나눠서 요청 (API 제한)
    rows = []
    for i in range(0, len(video_ids), 50):
        chunk = video_ids[i:i+50]
        resp = youtube.videos().list(
            id=','.join(chunk),
            part='snippet,statistics'
        ).execute()

        for item in resp.get('items', []):
            sn = item['snippet']
            st = item.get('statistics', {})
            rows.append({
                'video_id':     item['id'],
                'title':        sn['title'],
                'channel_name': sn['channelTitle'],
                'country':      country_map.get(item['id'], 'UNK'),
                'upload_date':  sn['publishedAt'][:10],
                'view_count':   int(st.get('viewCount', 0)),
                'like_count':   int(st.get('likeCount', 0)),
                'description':  sn.get('description', '')[:500],
            })

    return pd.DataFrame(rows)

videos_df = fetch_video_metadata(unique_targets)
print(f"\n📊 영상 메타데이터 수집: {len(videos_df)}개")
print(videos_df[['video_id', 'title', 'country', 'view_count']].to_string(index=False))

In [ ]:
# youtube_videos 테이블에 저장
if not videos_df.empty:
    upsert_df(videos_df, 'youtube_videos', conflict_cols=['video_id'])

## 댓글 수집 → youtube_comments

In [ ]:
def scrape_comments(video_id: str, max_comments: int = 200) -> pd.DataFrame:
    """댓글 수집 (댓글 비활성화 영상은 건너뜀)"""
    rows = []
    next_page_token = None
    try:
        while len(rows) < max_comments:
            resp = youtube.commentThreads().list(
                videoId=video_id, part='snippet',
                maxResults=min(100, max_comments - len(rows)),
                pageToken=next_page_token,
                textFormat='plainText', order='relevance'
            ).execute()

            for item in resp.get('items', []):
                c = item['snippet']['topLevelComment']['snippet']
                rows.append({
                    'video_id':     video_id,
                    'author':       c['authorDisplayName'],
                    'content':      c['textDisplay'],
                    'like_count':   c['likeCount'],
                    'comment_date': c['publishedAt'],
                })

            next_page_token = resp.get('nextPageToken')
            if not next_page_token:
                break
    except Exception as e:
        if 'commentsDisabled' in str(e):
            print(f"  ⚠️ 댓글 비활성화")
        else:
            print(f"  ❌ {e}")

    return pd.DataFrame(rows) if rows else pd.DataFrame()

# 전체 댓글 수집
all_comments = []
for vid in videos_df['video_id'].tolist():
    print(f"댓글: {vid}", end=' ')
    df = scrape_comments(vid, max_comments=200)
    if not df.empty:
        all_comments.append(df)
        print(f"→ {len(df)}개")
    else:
        print("→ 0개")

comments_df = pd.concat(all_comments, ignore_index=True) if all_comments else pd.DataFrame()
print(f"\n📊 총 댓글: {len(comments_df)}개")

In [ ]:
if not comments_df.empty:
    insert_df(comments_df, 'youtube_comments')

## 자막(STT) 수집 → youtube_stt

⚠️ **라오어 자막 우선순위**: `lo → vi → th → en → ko` 순으로 시도

In [ ]:
def scrape_transcript(video_id: str) -> pd.DataFrame:
    """자막 수집 (언어 우선순위: lo → vi → th → en → ko → 자동생성)"""
    lang_priority = ['lo', 'vi', 'th', 'en', 'ko']
    try:
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        transcript = None
        for lang in lang_priority:
            try:
                transcript = transcript_list.find_transcript([lang])
                break
            except:
                continue
        if transcript is None:
            transcript = transcript_list.find_generated_transcript(lang_priority)

        data = transcript.fetch()
        rows = [{
            'video_id':        video_id,
            'timestamp_start': item['start'],
            'timestamp_end':   item['start'] + item.get('duration', 0),
            'content':         item['text'].strip(),
        } for item in data if item['text'].strip()]
        return pd.DataFrame(rows)

    except TranscriptsDisabled:
        print(f"  ⚠️ 자막 비활성화")
        return pd.DataFrame()
    except Exception as e:
        print(f"  ❌ {e}")
        return pd.DataFrame()

all_stt = []
for vid in videos_df['video_id'].tolist():
    print(f"자막: {vid}", end=' ')
    df = scrape_transcript(vid)
    if not df.empty:
        all_stt.append(df)
        print(f"→ {len(df)} 세그먼트")
    else:
        print("→ 없음")

stt_df = pd.concat(all_stt, ignore_index=True) if all_stt else pd.DataFrame()
print(f"\n📊 총 자막 세그먼트: {len(stt_df)}개")

In [ ]:
if not stt_df.empty:
    insert_df(stt_df, 'youtube_stt')

## 최종 결과 확인

In [ ]:
from src.db import fetch_df

result = fetch_df("""
    SELECT
        v.video_id, v.title, v.country,
        COUNT(DISTINCT c.id) AS comments,
        COUNT(DISTINCT s.id) AS stt_segments
    FROM youtube_videos v
    LEFT JOIN youtube_comments c ON v.video_id = c.video_id
    LEFT JOIN youtube_stt s      ON v.video_id = s.video_id
    GROUP BY v.video_id, v.title, v.country
    ORDER BY comments DESC
""")
print(result.to_string(index=False))